In [1]:
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.nn import *
import torch
from tqdm import tqdm
from constants import *
from torch_geometric.data import Data, HeteroData
from torch_geometric.loader import DataLoader
from model_gnn import GNN
import gc

# DEVICE1/DEVICE2 imported from constants

In [2]:
ba_df = pd.read_parquet('data/ba_db.parquet')
ba_df['peptide'] = ['[CLS] ' + ' '.join(list(peptide)) + ' [SEP]' for peptide in ba_df['peptide'].values]
unique_alleles = ba_df['allele'].unique()
ba_df

,peptide,ba,allele
0,[CLS] A A A A A A N A A A M [SEP],0.117652,H2-K*b
1,[CLS] A A A A A A N A A A M [SEP],0.550545,H2-D*b
2,[CLS] A A A A N A A A A A M [SEP],0.415151,H2-K*b
3,[CLS] A A A A N A A A A A M [SEP],0.425625,H2-D*b
4,[CLS] A A A A N A A A M [SEP],0.212813,H2-K*b
...,...,...,...
208088,[CLS] Y H W S V S H Q L T G [SEP],0.010000,BoLA-6*041:01
208089,[CLS] Y L G K I D D Q [SEP],0.010000,HLA-C*08:03
208090,[CLS] Y L L E K L E E N F S [SEP],0.010000,BoLA-2*008:01
208091,[CLS] Y L L W L A L P F [SEP],0.010000,Eqca-1*60:01:01


In [3]:
mhc_df = pd.read_parquet('data/mhc_db.parquet')
mhc_df = mhc_df.loc[mhc_df['allele'].isin(unique_alleles)]
ba_df = ba_df.loc[ba_df['allele'].isin(mhc_df['allele'].values)]
unique_alleles = ba_df['allele'].unique()
mhc_df['sequence'] = ['[CLS] ' + ' '.join(list(sequence)) + ' [SEP]' for sequence in mhc_df['sequence'].values]
mhc_df

,allele,sequence
60,BoLA-1*021:01,[CLS] M G P R T L F V L L L G A L V L T E T R ...
61,BoLA-1*023:01,[CLS] M G P R T L F V L L L G A L A L T E T R ...
77,BoLA-2*008:01,[CLS] M R P R T L L L L L S G V L V L T E T L ...
80,BoLA-2*012:01,[CLS] M R P R T L L L L L S G V L V L T E T L ...
120,BoLA-3*001:01,[CLS] L L L S G V L V L T E T R A G S H S M R ...
...,...,...
9962,SLA-1*07:01,[CLS] M G P G A L F L L L S G T L A L T G T Q ...
9963,SLA-1*07:02,[CLS] M G P G A L F L L L S G T L A L T G T Q ...
10048,SLA-2*04:01,[CLS] M R V R G P Q A I L I L L S G A L A L T ...
10055,SLA-2*05:02,[CLS] M R V R G P Q A I L I L L S G A L A L T ...


In [4]:
MAX_LEN_MHC = mhc_df['sequence'].str.split().str.len().max()
MAX_LEN_PEP = ba_df['peptide'].str.split().str.len().max()
MAX_LEN_PEP, MAX_LEN_MHC

(np.int64(16), np.int64(374))

In [5]:
peptide_features = np.array([
    [TOKEN_VOCABULARY[e] for e in seq.split()] + [0 for _ in range(MAX_LEN_PEP - len(seq.split()))] for seq in ba_df['peptide'].values
])
mhc_features = np.array([
    [TOKEN_VOCABULARY[e] for e in seq.split()] + [0 for _ in range(MAX_LEN_MHC - len(seq.split()))] for seq in mhc_df['sequence'].values
])

ba_targets = np.expand_dims(ba_df['ba'].values, -1)

In [6]:
def get_hetero_data():
    data = []
    edge = torch.zeros((2, 1), dtype=torch.int32)
    for idx in tqdm(range(len(ba_df))):
        entry = HeteroData()
        entry['peptide'].x = torch.tensor(peptide_features[idx], dtype=torch.int32).unsqueeze(0)
        entry['mhc'].x = torch.tensor(mhc_features[mhc_df['allele'] == ba_df.iloc[idx]['allele']], dtype=torch.int32)
        entry['peptide', 'determines', 'mhc'].edge_index = edge
        entry['peptide', 'influences', 'peptide'].edge_index = edge
        entry['mhc', 'influences', 'mhc'].edge_index = edge
        entry['mhc'].y = torch.tensor(ba_targets[idx], dtype=torch.float32).unsqueeze(-1)
        data.append(entry)
    return data

data = get_hetero_data()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 183240/183240 [01:01<00:00, 2990.90it/s]


# Training

In [ ]:
gc.collect()
torch.cuda.empty_cache()


EPOCHS = 30
BATCH_SIZE = 200
train_loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True)

model = GNN(
    vocab_size=45,
    embedding_dim=128,
    dim_ff_enc_pep=128,
    n_heads_enc_pep=8,
    n_layers_enc_pep=6,
    dim_ff_enc_mhc=1024,
    n_heads_enc_mhc=8,
    n_layers_enc_mhc=6,
    n_heads_conv=8,
    n_layers_conv=2,
    dim_out_conv=32,
    dropout=0.1,
    act=F.leaky_relu,
    device1=DEVICE1,
    device2=DEVICE1,
).to(DEVICE1)
state_dict = torch.load('weights/pretrain_mhc.pt')
model.load_state_dict({key: value for key, value in state_dict.items() if 'embedding' in key}, strict=False)
model.encoder.mhc[0].load_state_dict({key[8:]: value for key, value in state_dict.items() if 'encoder' in key}, strict=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(1, EPOCHS+1):
    model.train()
    batches = tqdm(train_loader)
    bl = []
    for batch in batches:
        optimizer.zero_grad()
        out = model(batch)
        loss = F.binary_cross_entropy_with_logits(out, batch['mhc'].y.to(DEVICE1))
        bl.append(loss.item())
        loss.backward()
        optimizer.step()
        batches.set_description(f'Train epoch {epoch} - loss: {np.mean(bl[-100:]):.4f}')

torch.save(model.state_dict(), 'weights/pretrain_mhc_ba.pt')